# День 1 — токенизация

Модель: `distilbert-base-uncased`. Тексты Twitter на английском, колонки датасета: `clean_text` и `category` (−1 negative, 0 neutral, 1 positive).

`tokenizer.tokenize` показывает только куски текста. Полный вызов `tokenizer(...)` добавляет специальные токены `[CLS]` и `[SEP]`.

## Задача 2. Загрузка токенизатора

In [1]:
import sys
from pathlib import Path

import torch
from transformers import AutoTokenizer

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.tokenization import (
    MODEL_NAME,
    explain_tokenization,
    load_tokenizer,
    tokenize_texts,
)

tokenizer = load_tokenizer(MODEL_NAME)
print(AutoTokenizer.__name__, tokenizer.name_or_path)
print(tokenizer.vocab_size)
print(tokenizer.model_max_length)
print("torch", torch.__version__)

AutoTokenizer distilbert-base-uncased
30522
512
torch 2.14.1


## Задача 3. Токенизация текста

In [2]:
text = "This movie was absolutely amazing!"

tokens = tokenizer(text)
print(tokens)

input_ids = tokens["input_ids"]
print(f"Количество токенов: {len(input_ids)}")

decoded = tokenizer.decode(input_ids)
print(f"Декодировано: {decoded}")

{'input_ids': [101, 2023, 3185, 2001, 7078, 6429, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}
Количество токенов: 8
Декодировано: [CLS] this movie was absolutely amazing! [SEP]


## Задача 4. Работа с батчами

In [3]:
texts = [
    "This movie was great!",
    "Terrible movie, waste of time.",
]

tokens = tokenize_texts(texts, tokenizer)
print(f'Shape: {tokens["input_ids"].shape}')
print(f'Attention mask:\n{tokens["attention_mask"]}')

Shape: torch.Size([2, 9])
Attention mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])


## Задача 5. Специальные токены

`[CLS]` стоит в начале последовательности, `[SEP]` — в конце, `[PAD]` выравнивает короткие тексты в батче. В `attention_mask` у паддинга стоит 0.

In [4]:
print(f"CLS token: {tokenizer.cls_token} (ID: {tokenizer.cls_token_id})")
print(f"SEP token: {tokenizer.sep_token} (ID: {tokenizer.sep_token_id})")
print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

single = tokenizer(text, return_tensors="pt")
print(f'Input IDs: {single["input_ids"]}')
print(f'Decoded: {tokenizer.decode(single["input_ids"][0])}')

CLS token: [CLS] (ID: 101)
SEP token: [SEP] (ID: 102)
PAD token: [PAD] (ID: 0)
Input IDs: tensor([[ 101, 2023, 3185, 2001, 7078, 6429,  999,  102]])
Decoded: [CLS] this movie was absolutely amazing! [SEP]


## Задача 6. Функция explain_tokenization

Здесь токены без `[CLS]` и `[SEP]`: `tokenizer.tokenize` режет только слова.

In [5]:
explain_tokenization("Transformers are amazing!", tokenizer)

Исходный текст: Transformers are amazing!
Токены: ['transformers', 'are', 'amazing', '!']
IDs: [19081, 2024, 6429, 999]
Количество: 4


## Примеры из Twitter_Data.csv

Берём три непустые строки. Пустые `clean_text` и `category` пропускаем. Метки оставляем как в файле: −1, 0, 1.

In [6]:
import csv

LABELS = {"-1": "negative", "0": "neutral", "1": "positive"}
samples = []

with open(ROOT / "Twitter_Data.csv", newline="", encoding="utf-8") as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        sample_text = (row.get("clean_text") or "").strip()
        label = (row.get("category") or "").strip()
        if sample_text and label:
            samples.append((sample_text, label))
        if len(samples) == 3:
            break

for sample_text, label in samples:
    print(f"Метка category: {label} ({LABELS[label]})")
    explain_tokenization(sample_text, tokenizer)
    print()

batch = tokenize_texts([sample_text for sample_text, _ in samples], tokenizer)
print(f'Shape: {batch["input_ids"].shape}')
print(f'Attention mask:\n{batch["attention_mask"]}')

Метка category: -1 (negative)
Исходный текст: when modi promised “minimum government maximum governance” expected him begin the difficult job reforming the state why does take years get justice state should and not business and should exit psus and temples
Токены: ['when', 'mod', '##i', 'promised', '“', 'minimum', 'government', 'maximum', 'governance', '”', 'expected', 'him', 'begin', 'the', 'difficult', 'job', 'reforming', 'the', 'state', 'why', 'does', 'take', 'years', 'get', 'justice', 'state', 'should', 'and', 'not', 'business', 'and', 'should', 'exit', 'ps', '##us', 'and', 'temples']
IDs: [2043, 16913, 2072, 5763, 1523, 6263, 2231, 4555, 10615, 1524, 3517, 2032, 4088, 1996, 3697, 3105, 29455, 1996, 2110, 2339, 2515, 2202, 2086, 2131, 3425, 2110, 2323, 1998, 2025, 2449, 1998, 2323, 6164, 8827, 2271, 1998, 8436]
Количество: 37

Метка category: 0 (neutral)
Исходный текст: talk all the nonsense and continue all the drama will vote for modi
Токены: ['talk', 'all', 'the', 'nonsense', 'a